# 26-Class ECG Classifier V3.2 (PTB-XL + Chapman + Challenge + CODE-15%)

Trains ECGNetJoint on **~226K records** with transfer learning from V3 best (AUROC=0.9682).

---
| Cell | Runtime | Time | What |
|------|---------|------|------|
| 1 | **GPU/TPU** | ~60-75 min | Restore 4 datasets + setup |
| 2 | **GPU/TPU** | ~40-90 min (TPU) | Train V3.2 + save model to Drive |

**Before running:** extract  locally to  and wait for Drive to sync.

**Datasets:**
- PTB-XL (~2.7 GB tar, 18,524 records)
- Chapman (~5.5 GB tar, 42,390 records)
- Challenge 2021 (~7 GB tar, 50,842 records)
- CODE-15% parts 0-5 (~21 GB HDF5 direct download, ~115K records, ~5,750 AFIB)

**Why Drive API instead of FUSE mount?**  
FUSE caches every byte it reads to local disk. Streaming tars and downloading HDF5 files via Drive API bypasses that cache -- only the extracted/downloaded files (~36 GB) land on SSD.

In [ ]:
# -----------------------------------------------------------------------------
# Cell 1 -- GPU/TPU runtime | Setup + restore all data
#
# Tars required in Drive/EKG/ekg_datasets/:
#   challenge2021.tar (~7 GB)  chapman.tar (~5.5 GB)  ptbxl.tar (~2.7 GB)
#
# CODE-15% (HDF5 files): pre-extract code15.tar locally to
#   EKG/ekg_datasets/code15/raw/  ->  Drive syncs them automatically.
#   Cell downloads parts 0-5 (~21 GB) individually via Drive API.
#   Parts 6-17 skipped; dataset_code15.py indexes only what's present.
#
# ALL large files use Drive API directly (not FUSE mount) to avoid FUSE
# caching to local disk. Extracted/downloaded files (~36 GB) land on SSD.
#
# IMPORTANT: Do NOT import torch_xla -- it locks /dev/vfio/0.
# -----------------------------------------------------------------------------
import time, os, sys, subprocess, shutil
from pathlib import Path
t0 = time.time()
print('=' * 60)
print('Cell 1: GPU/TPU setup + restore data')
print('=' * 60)

# -- Verify accelerator ---------------------------------------------------
import torch
if os.path.exists('/dev/vfio/0'):
    accel = 'TPU'
elif torch.cuda.is_available():
    accel = f'GPU ({torch.cuda.get_device_name(0)})'
else:
    raise RuntimeError('No GPU or TPU! Runtime > Change runtime type > T4 GPU or TPU v6e')
print(f'Accelerator: {accel}')

# TPU: install torch_xla if needed (via subprocess only -- never import it here)
if accel == 'TPU':
    if subprocess.run([sys.executable, '-c', 'import torch_xla'],
                      capture_output=True).returncode != 0:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'torch_xla'], check=True)
        print('torch_xla installed')

# -- Mount Drive (for scripts + model only -- large files use Drive API) --
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'wfdb', 'scipy', 'scikit-learn', 'h5py'], check=True)

import shutil as _sh
free_gb = _sh.disk_usage('/content').free / 1e9
print(f'Deps ready | SSD free: {free_gb:.0f} GB (need ~36 GB) ({time.time()-t0:.0f}s)')

# -- Copy scripts from Drive ----------------------------------------------
SCRIPTS_DIR  = Path('/content/drive/MyDrive/EKG')
DRIVE_MODELS = Path('/content/drive/MyDrive/EKG/models')
os.chdir('/content')
os.makedirs('ekg_datasets', exist_ok=True)
os.makedirs('models', exist_ok=True)

for s in ['multilabel_v3.py', 'multilabel_classifier.py', 'cnn_classifier.py',
          'dataset_chapman.py', 'dataset_challenge.py', 'dataset_code15.py']:
    src = SCRIPTS_DIR / s
    if not src.exists():
        raise FileNotFoundError(f'{s} not found at {SCRIPTS_DIR}/')
    shutil.copy(src, f'/content/{s}')
print(f'Scripts copied ({time.time()-t0:.0f}s)')

# -- Restore model checkpoints for transfer learning ----------------------
# Priority: V3 best (AUROC=0.9682) > V2 > random init
for model_name in ['ecg_multilabel_v3_best.pt', 'ecg_multilabel_v3.pt']:
    dst = Path(f'/content/models/{model_name}')
    if not dst.exists():
        src = DRIVE_MODELS / model_name
        if src.exists():
            shutil.copy(src, dst)
            print(f'{model_name} copied for transfer learning ({dst.stat().st_size/1e6:.1f} MB)')

# -- Drive API auth -------------------------------------------------------
# authenticate_user() shows a one-time Google consent prompt per session.
# This is required -- drive.mount() credentials are separate from ADC.
from google.colab import auth as _colab_auth
_colab_auth.authenticate_user()
import google.auth
from google.auth.transport.requests import Request as _GReq
from googleapiclient.discovery import build as _build
import requests as _rq

_creds, _ = google.auth.default(
    scopes=[
        'https://www.googleapis.com/auth/drive.readonly',
        'https://www.googleapis.com/auth/gmail.send',
    ]
)
_creds.refresh(_GReq())
_svc = _build('drive', 'v3', credentials=_creds, cache_discovery=False)
_gmail_svc = _build('gmail', 'v1', credentials=_creds, cache_discovery=False)
_my_email = _gmail_svc.users().getProfile(userId='me').execute()['emailAddress']
print(f'Gmail ready: will notify {_my_email}')

# -- Drive file lookup (direct search -- no folder-tree navigation) -------
def _find_drive_file(name):
    # Find file by exact name anywhere in Drive (handles shortcuts + shared drives).
    q = f"name='{name}' and trashed=false"
    r = _svc.files().list(
        q=q, fields='files(id,size)',
        supportsAllDrives=True, includeItemsFromAllDrives=True
    ).execute()
    files = r.get('files', [])
    if not files: return None, 0
    return files[0]['id'], int(files[0].get('size', 0))

# Verify Drive API works by checking FUSE mount is accessible
_fuse_check = Path('/content/drive/MyDrive/EKG/ekg_datasets')
if not _fuse_check.exists():
    raise FileNotFoundError(
        'Drive/EKG/ekg_datasets/ not found via FUSE mount.\n'
        'Check that Google Drive for Desktop has synced the EKG folder.'
    )
print(f'Drive API ready | FUSE verified: {_fuse_check}')

def stream_tar(fid, target, label, size_gb):
    # Stream tar from Drive API -> tar stdin. No FUSE caching.
    _creds.refresh(_GReq())
    print(f'{label}: streaming {size_gb:.1f} GB ...', flush=True)
    url = f'https://www.googleapis.com/drive/v3/files/{fid}?alt=media'
    hdrs = {'Authorization': f'Bearer {_creds.token}'}
    proc = subprocess.Popen(
        ['tar', 'xf', '-', '-C', target],
        stdin=subprocess.PIPE, stderr=subprocess.PIPE
    )
    streamed = 0
    total = size_gb * 1e9
    last_pct = -1
    try:
        with _rq.get(url, headers=hdrs, stream=True, timeout=(30, 600)) as resp:
            resp.raise_for_status()
            for chunk in resp.iter_content(chunk_size=32 << 20):  # 32 MB chunks
                if chunk:
                    proc.stdin.write(chunk)
                    streamed += len(chunk)
                    pct = int(streamed * 100 / total) if total > 0 else 0
                    if pct >= last_pct + 10:
                        last_pct = pct
                        elapsed = time.time() - t0
                        free = _sh.disk_usage('/content').free / 1e9
                        print(f'  {label}: {pct}% ({streamed/1e9:.1f}/{size_gb:.1f} GB, {elapsed:.0f}s, {free:.0f} GB free)', flush=True)
    except BrokenPipeError:
        pass  # tar exited early -- fall through to check returncode + stderr
    finally:
        # Always close stdin; runs even when BrokenPipeError is caught.
        try:
            proc.stdin.close()
        except Exception:
            pass
    # Use stderr.read() + wait() instead of communicate() --
    # communicate() flushes stdin and raises ValueError on Python 3.12+ if stdin is closed.
    err = proc.stderr.read()
    proc.wait()
    if proc.returncode:
        raise RuntimeError(
            f'{label}: tar failed (exit {proc.returncode})\n'
            f'stderr: {err.decode()[:500]}\n'
            f'Streamed: {streamed/1e9:.1f} GB | SSD free: {_sh.disk_usage("/content").free/1e9:.0f} GB'
        )
    print(f'{label}: done ({time.time()-t0:.0f}s)', flush=True)

def _download_file(fid, dest_path, label, size_gb):
    # Download a single file from Drive API directly to dest_path.
    # Uses a .tmp file so a failed download never leaves a partial result.
    _creds.refresh(_GReq())
    dest_path = Path(dest_path)
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest_path.with_suffix(dest_path.suffix + '.tmp')
    print(f'{label}: {size_gb:.2f} GB ...', flush=True)
    url = f'https://www.googleapis.com/drive/v3/files/{fid}?alt=media'
    hdrs = {'Authorization': f'Bearer {_creds.token}'}
    downloaded = 0
    total = size_gb * 1e9
    last_pct = -1
    try:
        with _rq.get(url, headers=hdrs, stream=True, timeout=(30, 600)) as resp:
            resp.raise_for_status()
            with open(tmp, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=32 << 20):  # 32 MB chunks
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
                        pct = int(downloaded * 100 / total) if total > 0 else 0
                        if pct >= last_pct + 10:
                            last_pct = pct
                            elapsed = time.time() - t0
                            free = _sh.disk_usage('/content').free / 1e9
                            print(f'  {label}: {pct}% ({downloaded/1e9:.1f}/{size_gb:.1f} GB, {elapsed:.0f}s, {free:.0f} GB free)', flush=True)
    except Exception:
        tmp.unlink(missing_ok=True)
        raise
    tmp.rename(dest_path)
    print(f'{label}: done ({time.time()-t0:.0f}s)', flush=True)

# -- Restore datasets -----------------------------------------------------

# [1/4] CODE-15% parts 0-5 (~21 GB, ~115K records, ~5750 AFIB records)
# Requires code15.tar pre-extracted locally to EKG/ekg_datasets/code15/raw/
# so Drive syncs the 18 HDF5 files individually. Only parts 0-5 downloaded.
C15_FUSE = Path('/content/drive/MyDrive/EKG/ekg_datasets/code15/raw')
C15_RAW  = Path('/content/ekg_datasets/code15/raw')
C15_RAW.mkdir(parents=True, exist_ok=True)

if not C15_FUSE.exists():
    print('CODE-15%: raw/ folder not found on Drive -- SKIPPED')
    print('  Extract code15.tar to EKG/ekg_datasets/code15/raw/ and re-sync Drive.')
else:
    # exams.csv is small (~5 MB) -- FUSE copy is fine
    _exams_dst = C15_RAW / 'exams.csv'
    if not _exams_dst.exists():
        shutil.copy(C15_FUSE / 'exams.csv', _exams_dst)
        print(f'CODE-15% exams.csv: copied ({time.time()-t0:.0f}s)')
    else:
        print('CODE-15% exams.csv: already on SSD -- skipping')

    # Parts 0-5: Drive API download (bypasses FUSE cache, writes directly to SSD)
    for _i in range(6):
        _fname = f'exams_part{_i}.hdf5'
        _dst   = C15_RAW / _fname
        if _dst.exists():
            print(f'CODE-15% {_fname}: already on SSD -- skipping')
            continue
        if not (C15_FUSE / _fname).exists():
            print(f'CODE-15% {_fname}: not on Drive -- skipping part {_i}')
            continue
        _fid, _sz = _find_drive_file(_fname)
        if not _fid:
            # Fallback: FUSE copy (uses FUSE cache but still works)
            print(f'CODE-15% {_fname}: Drive API lookup failed, falling back to FUSE copy')
            shutil.copy(C15_FUSE / _fname, _dst)
        else:
            _download_file(_fid, _dst, f'CODE-15% [{_i+1}/6] {_fname}', _sz / 1e9)

    # Build index (reads all present HDF5 parts, silently skips missing ones)
    _c15_index = Path('/content/ekg_datasets/code15/code15_index.csv')
    if not _c15_index.exists():
        print('CODE-15%: building index (~5-15 min) ...', flush=True)
        subprocess.run([sys.executable, 'dataset_code15.py', '--index'], check=True)
    else:
        _nrec = sum(1 for _ in open(_c15_index)) - 1
        print(f'CODE-15% index: {_nrec:,} records -- skipping rebuild')

# [2/4] Challenge 2021 (~7 GB) -- optional
challenge_dir = Path('/content/ekg_datasets/challenge2021')
if len(list(challenge_dir.rglob('*.mat'))) >= 50000:
    print('Challenge: already on SSD -- skipping')
else:
    fid, sz = _find_drive_file('challenge2021.tar')
    if fid:
        stream_tar(fid, '/content/ekg_datasets', 'Challenge', sz / 1e9)
    else:
        print('WARNING: challenge2021.tar not on Drive -- skipping (optional)')

# [3/4] Chapman (~5.5 GB, paths inside: ekg_datasets/... -> extract -C /content)
chapman_dir = Path('/content/ekg_datasets/chapman')
if len(list(chapman_dir.rglob('*.mat'))) >= 44000:
    print('Chapman: already on SSD -- skipping')
else:
    fid, sz = _find_drive_file('chapman.tar')
    if not fid:
        raise FileNotFoundError('chapman.tar not found in Drive.')
    stream_tar(fid, '/content', 'Chapman', sz / 1e9)

# [4/4] PTB-XL (~2.7 GB)
ptbxl_dir = Path('/content/ekg_datasets/ptbxl')
if len(list(ptbxl_dir.rglob('*.dat'))) >= 18000:
    print('PTB-XL: already on SSD -- skipping')
else:
    fid, sz = _find_drive_file('ptbxl.tar')
    if not fid:
        raise FileNotFoundError('ptbxl.tar not found in Drive.')
    stream_tar(fid, '/content/ekg_datasets', 'PTB-XL', sz / 1e9)

free_after = _sh.disk_usage('/content').free / 1e9
print(f'\nCell 1 done ({time.time()-t0:.0f}s) | SSD free: {free_after:.0f} GB | ready to train')


Cell 1: GPU/TPU setup + restore data
Accelerator: GPU (NVIDIA RTX PRO 6000 Blackwell Server Edition)
Mounted at /content/drive
Deps ready | SSD free: 43 GB (need ~36 GB) (5s)
Scripts copied (5s)
Drive API ready | FUSE verified: /content/drive/MyDrive/EKG/ekg_datasets
CODE-15% exams.csv: already on SSD -- skipping
CODE-15% exams_part0.hdf5: already on SSD -- skipping
CODE-15% exams_part1.hdf5: already on SSD -- skipping
CODE-15% exams_part2.hdf5: already on SSD -- skipping
CODE-15% exams_part3.hdf5: already on SSD -- skipping
CODE-15% exams_part4.hdf5: already on SSD -- skipping
CODE-15% exams_part5.hdf5: already on SSD -- skipping
CODE-15% index: 120,000 records -- skipping rebuild
Challenge: already on SSD -- skipping
Chapman: streaming 5.5 GB ...
  Chapman: 9% (0.5/5.5 GB, 16s, 43 GB free)
  Chapman: 19% (1.1/5.5 GB, 24s, 42 GB free)
  Chapman: 29% (1.6/5.5 GB, 35s, 42 GB free)
  Chapman: 39% (2.2/5.5 GB, 47s, 41 GB free)
  Chapman: 49% (2.7/5.5 GB, 58s, 41 GB free)
  Chapman: 59% (3

In [ ]:
# -----------------------------------------------------------------------------
# Cell 2 -- GPU/TPU runtime | Train V3.2 + save to Drive
# TPU (v6e): ~40-90 min  |  GPU (T4): ~2-4 hrs
# First TPU epoch is slow (XLA compilation) -- this is normal.
# IMPORTANT: Do NOT import torch_xla -- it locks /dev/vfio/0.
# -----------------------------------------------------------------------------
import time, os, sys, subprocess, shutil, re
from pathlib import Path
t0 = time.time()
print('=' * 60)
print('Cell 2: Train V3.2 + save model')
print('=' * 60)
os.chdir('/content')

import torch
if os.path.exists('/dev/vfio/0'):
    accel = 'TPU'
elif torch.cuda.is_available():
    accel = f'GPU ({torch.cuda.get_device_name(0)})'
else:
    raise RuntimeError('No GPU or TPU! Run Cell 1 first.')

if accel == 'TPU':
    try:
        fd = os.open('/dev/vfio/0', os.O_RDONLY | os.O_NONBLOCK)
        os.close(fd)
    except OSError as e:
        raise RuntimeError(
            f'TPU device busy ({e}). Runtime > Restart runtime, then re-run Cell 1 + 2.'
        )

print(f'Accelerator: {accel}')
print('Starting multilabel_v3.py...', flush=True)
print('-' * 60, flush=True)

proc = subprocess.Popen(
    [sys.executable, '-u', 'multilabel_v3.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
_captured = []
for line in proc.stdout:
    print(line, end='', flush=True)
    _captured.append(line.rstrip())
proc.wait()

print('-' * 60, flush=True)
if proc.returncode != 0:
    raise RuntimeError(f'Training failed (exit {proc.returncode})')

src = Path('/content/models/ecg_multilabel_v3.pt')
DRIVE_MODELS = Path('/content/drive/MyDrive/EKG/models')
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
if src.exists():
    dst = DRIVE_MODELS / 'ecg_multilabel_v3.pt'
    shutil.copy(src, dst)
    print(f'Model saved to Drive ({dst.stat().st_size/1e6:.0f} MB)')
    print('Syncs to local PC via Google Drive for Desktop.')
else:
    print('WARNING: model file not found')

# -- Send completion email -----------------------------------------------
try:
    import base64
    from email.mime.text import MIMEText

    # Collect key summary lines from training output
    _kw = ('MacroAUROC', 'MacroF1', 'MicroF1', 'Early stop',
           'Checkpoint saved', '  Ep ')
    _summary = [l for l in _captured if any(k in l for k in _kw)]
    # Per-class breakdown lines (4-space indent + label + F1=)
    _summary += [l for l in _captured if re.match(r'    \w', l) and 'F1=' in l]
    _summary = list(dict.fromkeys(_summary))  # dedup, preserve order

    # Extract MacroAUROC for subject line
    _mac = next((l for l in reversed(_captured) if 'MacroAUROC' in l), '')
    _av = re.search(r'[\d.]{5,}', _mac)
    _auroc_str = _av.group() if _av else '?'

    _elapsed = f'{(time.time()-t0)/60:.0f} min'
    _body = (
        f'Training completed in {_elapsed} on {accel}.\n\n'
        + '\n'.join(_summary[-60:])
        + f'\n\nFull run: {len(_captured)} lines.\n'
    )
    _msg = MIMEText(_body)
    _msg['To'] = _my_email
    _msg['From'] = _my_email
    _msg['Subject'] = f'[EKG] Training complete - MacroAUROC={_auroc_str}'
    _raw = base64.urlsafe_b64encode(_msg.as_bytes()).decode()
    _creds.refresh(_GReq())
    _gmail_svc.users().messages().send(userId='me', body={'raw': _raw}).execute()
    print(f'Email sent to {_my_email}')
except Exception as _e:
    print(f'Email notification failed (non-fatal): {_e}')

print(f'\nCell 2 done ({time.time()-t0:.0f}s)')


Cell 2: Train V3.2 + save model
Accelerator: GPU (NVIDIA RTX PRO 6000 Blackwell Server Edition)
Starting multilabel_v3.py...
------------------------------------------------------------

  V3 Multi-Label ECG Classifier  (26 conditions)
  PTB-XL + Chapman + PhysioNet 2021 Challenge + CODE-15%
  Device: GPU (NVIDIA RTX PRO 6000 Blackwell Server Edition)
  Batch size: 64
  Learning rate: 3.0e-04 (scaled from 3.0e-04)
  Indexing 21799 records for multi-label...
  Kept 18524 records  (skipped: 3275 no-label, 0 no-file)
  Per-class positives:  NORM=9438  PVC=1027  LVH=1751  IMI=1714  ASMI=2007  CLBBB=536  CRBBB=540  LAFB=1622  1AVB=790  ISC_=1260  NDT=1824  IRBBB=1115
Chapman-Shaoxing: 42390 records loaded
Loading Challenge datasets...
  [challenge] georgia         :  10045 kept  (skipped: 238 no-label, 34 no-hea)
  [challenge] cpsc_2018       :   5634 kept  (skipped: 153 no-label, 46 no-hea)
  [challenge] cpsc_2018_extra :   2461 kept  (skipped: 925 no-label, 27 no-hea)
  [challenge] ningbo